# 01 · Data Source and Quality

Verify what was supplied, identify structural limitations and define which records are safe for each stage of the analysis.

## Reading guide

This notebook is part of a connected workflow. It states the decision being made, shows the supporting checks and records limitations alongside the result. Source files are never modified in place.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("FIFAR_DATA_DIR", PROJECT_ROOT / "data" / "raw" / "FiFAR"))
REPORTS = PROJECT_ROOT / "reports"
IMAGES = PROJECT_ROOT / "images"

sns.set_theme(style="whitegrid")
CORAL = "#F08FA0"
TEAL = "#0E6268"
DARK = "#15262B"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Set FIFAR_DATA_DIR to the extracted official FiFAR directory before running this notebook."
    )

## 1. Source files

FiFAR contains a base bank-account application table, author-generated risk scores, a selected alert population, synthetic analyst decisions and capacity scenarios. These components answer different questions and must not be mixed without checking their keys and row order.

In [ ]:
paths = {
    "base": DATA_ROOT / "alert_data" / "Base.csv",
    "scores": DATA_ROOT / "alert_data" / "processed_data" / "BAF_alert_model_score.parquet",
    "alerts": DATA_ROOT / "alert_data" / "processed_data" / "alerts.parquet",
    "expert_predictions": DATA_ROOT / "synthetic_experts" / "expert_predictions.parquet",
    "expert_parameters": DATA_ROOT / "synthetic_experts" / "expert_parameters.parquet",
}
pd.DataFrame({"file": paths.keys(), "exists": [path.exists() for path in paths.values()]})

## 2. Base schema and target

In [ ]:
base = pd.read_csv(paths['base'])
base.shape, base.head(3)

In [ ]:
schema = pd.DataFrame({
    "dtype": base.dtypes.astype(str),
    "missing": base.isna().sum(),
    "unique": base.nunique(dropna=False),
})
schema

## 3. Structural completeness

A value of `-1` is a documented source sentinel in several fields. It is not the same as a truncated CSV row and must not be replaced without a field-specific decision.

In [ ]:
quality = pd.Series({
    "rows": len(base),
    "columns": base.shape[1],
    "duplicate_rows": base.duplicated().sum(),
    "missing_cells": base.isna().sum().sum(),
    "incomplete_rows": base.isna().any(axis=1).sum(),
    "fraud_cases": base["fraud_bool"].sum(),
    "fraud_rate": base["fraud_bool"].mean(),
})
quality

In [ ]:
base.loc[base.isna().any(axis=1)]

**Decision.** Remove only the final structurally incomplete row. Preserve documented sentinels and report their frequency separately.

In [ ]:
base_complete = base.loc[~base.isna().any(axis=1)].copy()
base_complete.shape

## 4. Monthly completeness

In [ ]:
monthly = base_complete.groupby("month")["fraud_bool"].agg(
    applications="size", fraud_cases="sum", fraud_rate="mean"
)
monthly

In [ ]:
fig, left = plt.subplots(figsize=(11, 5))
right = left.twinx()
left.bar(monthly.index, monthly["applications"], color=TEAL, alpha=.82)
right.plot(monthly.index, monthly["fraud_rate"] * 100, color=CORAL, marker="o", linewidth=2.5)
left.axvspan(3.65, 4.35, color=CORAL, alpha=.12)
left.set(xlabel="Source month", ylabel="Applications", title="Volume and observed fraud rate")
right.set_ylabel("Fraud rate (%)")
plt.show()

Month 4 has substantially lower volume and an implausibly elevated observed fraud rate. Because the file terminates during this month, it is excluded from primary temporal comparisons rather than treated as genuine distribution shift.

## 5. Related table alignment

In [ ]:
alerts = pd.read_parquet(paths["alerts"])
scores = pd.read_parquet(paths["scores"])
experts = pd.read_parquet(paths["expert_predictions"])
pd.DataFrame({
    "table": ["base", "author scores", "alerts", "expert predictions"],
    "rows": [len(base), len(scores), len(alerts), len(experts)],
    "columns": [base.shape[1], scores.shape[1], alerts.shape[1], experts.shape[1]],
})

The expert-prediction table has the same row count as the alerts table. Their row alignment is used only after confirming this invariant. The scored table is a separate author-provided population and is not joined by row number to the base CSV.

## Conclusion

The archive supports a fraud-ranking study and a downstream alert-review study. The incomplete base month is a material limitation, but it can be isolated without fabricating records or values.